# Phase 1 — Inspect Dataset Structure and Verify Splits

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import json
DATA = "/content/drive/MyDrive/Data"        # drive url
def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
def key(ex): return json.dumps(ex["messages"][1:], sort_keys=True)
train = load_jsonl(f"{DATA}/train_agr_fixed.jsonl")
val   = load_jsonl(f"{DATA}/val_agr_fixed.jsonl")
test  = load_jsonl(f"{DATA}/test_agr_fixed.jsonl")
print("loaded:", len(train), len(val), len(test))

# Phase 2 — Dataset Validation and Quality Checks

In [ ]:
def check(ex):
    m = ex["messages"]
    roles = [x["role"] for x in m]
    if roles[0] != "system": return "missing system prompt"
    if roles[1:] != ["user", "assistant"] * ((len(roles) - 1) // 2): return "bad turn order"
    if any(not x["content"].strip() for x in m): return "empty content"

for name, data in [("train", train), ("val", val), ("test", test)]:
    bad = [(i, err) for i, ex in enumerate(data) if (err := check(ex))]
    print(name, "->", len(bad), "problems:", bad[:5])

In [ ]:
import random
prompts = {ex["messages"][0]["content"] for ex in train + val + test}
print("Unique system prompts:", len(prompts), "(should be 1)")
for ex in random.sample(train, 3):
    for m in ex["messages"][1:]:
        print(f"[{m['role']}] {m['content'][:150]}")
    print("-" * 50)

Leakage and duplicate check

In [ ]:
import json
def key(ex): return json.dumps(ex["messages"][1:], sort_keys=True)
sets = {n: {key(e) for e in d} for n, d in [("train", train), ("val", val), ("test", test)]}
print("train ∩ val :", len(sets["train"] & sets["val"]))
print("train ∩ test:", len(sets["train"] & sets["test"]))
print("val ∩ test  :", len(sets["val"] & sets["test"]))
for n, d in [("train", train), ("val", val), ("test", test)]:
    print(n, "internal duplicates:", len(d) - len(sets[n]))

# Phase 3 — Dataset Statistics

**Two reasons:**

**1. Set the right `max_seq_length`:**
Conversation lengths help us choose a value that avoids cutting off messages and avoids wasting GPU memory.

**2. Check repeated greetings:**
We measure how often greetings appear in the dataset to see if they may cause the model to repeat them too often.


In [ ]:
import statistics
convs = train + val + test
turns = [sum(1 for m in ex["messages"] if m["role"] == "user") for ex in convs]
words = [sum(len(m["content"].split()) for m in ex["messages"]) for ex in convs]
a_words = [len(m["content"].split()) for ex in convs for m in ex["messages"] if m["role"] == "assistant"]
print("user turns: min", min(turns), "| median", statistics.median(turns), "| max", max(turns))
print("single-turn:", sum(t == 1 for t in turns), "| multi-turn:", sum(t > 1 for t in turns))
print("words per conv (incl. system): median", statistics.median(words), "| max", max(words))
print("words per assistant reply: median", statistics.median(a_words), "| max", max(a_words))
print("longest conv ≈", int(max(words) * 1.3), "tokens (rough estimate)")

In [ ]:
import re
greet = re.compile(r"^(hello|hi|dear|greetings)\b", re.I)
first = sum(1 for ex in convs if greet.match(ex["messages"][2]["content"]))
later = sum(1 for ex in convs for m in ex["messages"][4::2] if greet.match(m["content"]))
print("conversations:", len(convs))
print("greeting opens the FIRST assistant reply:", first, "(fine, even polite)")
print("greetings in LATER assistant replies:", later, "(the bad habit)")

# Phase 4 — Install QLoRA Dependencies

In [ ]:
!pip -q install -U transformers peft trl bitsandbytes accelerate datasets wandb

verify GPU and versions

In [ ]:
import torch, transformers, peft, trl, bitsandbytes, accelerate, datasets, wandb
print("GPU:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT ENABLED")
for lib in (transformers, peft, trl, bitsandbytes, accelerate, datasets, wandb):
    print(lib.__name__, lib.__version__)

# Phase 5 — Prepare Dataset for Qwen Fine-Tuning

**Load the Qwen tokenizer and measure token lengths to choose `max_seq_length` for training.**


Block A — load tokenizer + render one conversation as ChatML

In [ ]:
import json
from transformers import AutoTokenizer
DATA = "/content/drive/MyDrive/Data"   # same folder as before
load = lambda p: [json.loads(l) for l in open(f"{DATA}/{p}", encoding="utf-8") if l.strip()]
train, val, test = load("train_agr_fixed.jsonl"), load("val_agr_fixed.jsonl"), load("test_agr_fixed.jsonl")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-1.8B-Chat")
print("loaded:", len(train), len(val), len(test))
print(tok.apply_chat_template(train[0]["messages"], tokenize=False)[:400])

Block B — measure real token lengths:

In [ ]:
lens = sorted(len(tok.apply_chat_template(ex["messages"], tokenize=True)) for ex in train + val + test)
print("tokens -> median:", lens[len(lens)//2], "| 95th:", lens[int(len(lens)*0.95)], "| max:", lens[-1])
print("conversations over 1024 tokens:", sum(l > 1024 for l in lens))

# Phase 6 — Configure Training Parameters

Define three configuration objects — quantization (4-bit), LoRA (the trainable adapter), and training hyperparameters — without loading the model yet.

Block A — quantization  + LoRA adapter :

In [ ]:
import torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,        # T4 has no bf16
    bnb_4bit_use_double_quant=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM", target_modules="all-linear")

Block B — training hyperparameters (SFTConfig):

In [ ]:
from trl import SFTConfig
cfg = SFTConfig(
    output_dir="qwen-cs-lora", max_length=1024, num_train_epochs=1,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,    #  batch 1
    learning_rate=4e-5, bf16=True, optim="paged_adamw_8bit",
    loss_type="chunked_nll",                                          # low-memory loss
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10, eval_strategy="epoch", save_strategy="epoch", report_to="wandb")
print("batch:", cfg.per_device_train_batch_size, "| accum:", cfg.gradient_accumulation_steps, "| loss:", cfg.loss_type)

# Phase 7 — Fine-Tune Qwen 1.5 with QLoRA

Load Qwen in 4-bit, attach the LoRA adapter, and run training while W&B logs the loss curves.

Block A — log in to W&B and load the 4-bit model:

In [ ]:
import os , wandb
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_PROJECT"] = "qwen-cs-finetune"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = "Qwen/Qwen1.5-1.8B-Chat"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={"": 0}, dtype=torch.bfloat16)
print("✅ base model loaded in bf16 + 4-bit")

Block B — build the trainer and reveal how little we train:

In [ ]:
import json
from trl import SFTTrainer
from datasets import Dataset
DATA = "/content/drive/MyDrive/Data"
ld = lambda p: [json.loads(l) for l in open(f"{DATA}/{p}", encoding="utf-8") if l.strip()]
trainer = SFTTrainer(model=model, args=cfg,
    train_dataset=Dataset.from_list(ld("train_agr_fixed.jsonl")),
    eval_dataset=Dataset.from_list(ld("val_agr_fixed.jsonl")),
    processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()

Block C — train, then save the adapter locally:

In [ ]:
trainer.train()
trainer.save_model("qwen-cs-lora-final")
print("✅ training done — adapter saved to qwen-cs-lora-final")

# Phase 8 — Save Adapters & Checkpoints

Block A — save to Drive :

In [ ]:
import os
SAVE = "/content/drive/MyDrive/Data/qwen-cs-lora-final"
trainer.save_model(SAVE)        # adapter weights + config
tok.save_pretrained(SAVE)       # tokenizer alongside it
print("✅ saved to Drive:", sorted(os.listdir(SAVE)))

Block B — upload to Hugging Face: (optional)

# Phase 9 — Base-Model Inference (the baseline)

Block A — load the base model (no adapter):

In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
DATA = "/content/drive/MyDrive/Data"
ld = lambda p: [json.loads(l) for l in open(f"{DATA}/{p}", encoding="utf-8") if l.strip()]
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
M = "Qwen/Qwen1.5-1.8B-Chat"
tok = AutoTokenizer.from_pretrained(M)
base = AutoModelForCausalLM.from_pretrained(M, quantization_config=bnb, device_map={"": 0}, dtype=torch.bfloat16)
print("✅ base model loaded")

Block B — generate answers for all 28 test questions and save:

In [ ]:
def gen(model, msgs, n=256):
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=n, do_sample=False)   # greedy = reproducible
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

test = ld("test_agr_fixed.jsonl")
rows = [{"prompt": ex["messages"][:-1], "reference": ex["messages"][-1]["content"],
         "base": gen(base, ex["messages"][:-1])} for ex in test]
json.dump(rows, open(f"{DATA}/bench_base.json", "w"), ensure_ascii=False, indent=2)
print("done:", len(rows), "| sample base answer:\n", rows[0]["base"][:300])

# Phase 10 — Fine-Tuned Inference

Block A — attach your adapter to the base model:

In [ ]:
from peft import PeftModel
ADAPTER = "/content/drive/MyDrive/Data/qwen-cs-lora-final"
ft = PeftModel.from_pretrained(base, ADAPTER)
print("✅ fine-tuned model ready (base + adapter)")

Block B — generate fine-tuned answers and save the combined file:

In [ ]:
rows = json.load(open(f"{DATA}/bench_base.json"))
for r in rows:
    r["ft"] = gen(ft, r["prompt"])                 # same gen() as base = fair
json.dump(rows, open(f"{DATA}/bench_all.json", "w"), ensure_ascii=False, indent=2)
print("done:", len(rows))
print("Q   :", rows[0]["prompt"][-1]["content"][:150])
print("BASE:", rows[0]["base"][:200])
print("FT  :", rows[0]["ft"][:200])

# Phase 11 — ChatGPT as LLM Judge

Block A — connect to OpenAI (uses getpass so your key isn't saved in the notebook):

In [ ]:
!pip -q install -U openai

from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
print("OpenAI client ready")

Block B — define the blind judge:

In [ ]:
import json, random, statistics
DATA = "/content/drive/MyDrive/Data"
rows = json.load(open(f"{DATA}/bench_all.json"))
DIMS = ["helpfulness","accuracy","policy_alignment","tone","safety","escalation","conciseness"]
SYS = ("Grade two agritech customer-service replies (A, B) 1-5 on each: " + ", ".join(DIMS) +
  ". Rules: accuracy=no invented products/dosages; policy_alignment=ask if unsure, never guess; "
  'conciseness=direct not rambling. Return ONLY JSON {"A":{dim:int,...},"B":{dim:int,...}}.')
fmt = lambda m: "\n".join(f"{x['role']}: {x['content']}" for x in m if x['role']!='system')
def judge(r):
    flip = random.random() < 0.5
    a, b = (r["ft"], r["base"]) if flip else (r["base"], r["ft"])
    u = f"QUESTION:\n{fmt(r['prompt'])}\n\nIDEAL REFERENCE:\n{r['reference']}\n\nRESPONSE A:\n{a}\n\nRESPONSE B:\n{b}"
    o = client.chat.completions.create(model="gpt-4o-mini", temperature=0,
        response_format={"type":"json_object"}, messages=[{"role":"system","content":SYS},{"role":"user","content":u}])
    s = json.loads(o.choices[0].message.content)
    return (s["B"], s["A"]) if flip else (s["A"], s["B"])     # always (base, ft)

Block C — judge all 28 and preview averages:

In [ ]:
for r in rows:
    r["base_scores"], r["ft_scores"] = judge(r)
json.dump(rows, open(f"{DATA}/bench_judged.json", "w"), ensure_ascii=False, indent=2)
for label, k in [("BASE", "base_scores"), ("FT", "ft_scores")]:
    print(label, {d: round(statistics.mean(x[k][d] for x in rows), 2) for d in DIMS})

# Phase 12 — Benchmark Comparison Report

Block A — comparison table + CSV:

In [ ]:
import json, statistics, pandas as pd
DATA = "/content/drive/MyDrive/Data"
rows = json.load(open(f"{DATA}/bench_judged.json"))
DIMS = ["helpfulness","accuracy","policy_alignment","tone","safety","escalation","conciseness"]
m = lambda k,d: round(statistics.mean(r[k][d] for r in rows),2)
df = pd.DataFrame({"Base":[m("base_scores",d) for d in DIMS], "FineTuned":[m("ft_scores",d) for d in DIMS]}, index=DIMS)
df["Delta"] = (df["FineTuned"]-df["Base"]).round(2)
df["Winner"] = ["FT" if x>0 else "Base" if x<0 else "tie" for x in df["Delta"]]
df.loc["OVERALL"] = [df["Base"].mean().round(2), df["FineTuned"].mean().round(2), df["Delta"].mean().round(2), ""]
print(df.to_string()); df.to_csv(f"{DATA}/benchmark_table.csv")

Block B — bar chart + win count:

In [ ]:
import matplotlib.pyplot as plt
d = df.drop("OVERALL")
ax = d[["Base","FineTuned"]].plot.bar(figsize=(9,4), rot=30)
ax.set_title("Base vs Fine-Tuned — LLM Judge Scores (1-5)"); ax.set_ylabel("Mean score"); ax.set_ylim(0,5)
plt.tight_layout(); plt.savefig(f"{DATA}/benchmark_chart.png", dpi=120, bbox_inches="tight"); plt.show()
print(f"FT wins {(d['Winner']=='FT').sum()}/7 dimensions | Overall: FT {df.loc['OVERALL','FineTuned']} vs Base {df.loc['OVERALL','Base']}")

# Phase 13 — Simple StreamLit chatbot

In [ ]:
%%writefile app.py
import streamlit as st, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

st.set_page_config(page_title="BAMBOO TUNE", page_icon="🌱", layout="wide")

# ============================================================
# BACKEND - fine-tuned model
# ============================================================
DATA = "/content/drive/MyDrive/Data"
BASE_MODEL = "Qwen/Qwen1.5-1.8B-Chat"
ADAPTER = f"{DATA}/qwen-cs-lora-final"
SYS = """
You are a customer-service assistant for an agritech company.

Response style:
- Maximum 2 sentences.
- Maximum 50 words.
- Answer only the user's question.
- Do not explain background science unless asked.
- Do not provide educational articles.
- Do not repeat information.
- If information is missing, ask one short clarifying question.
- Include dosage, mixing, timing, compatibility, or safety details when available.
- Never invent information.

Good example:
User: Can overdosing pesticide be dangerous?
Assistant: Yes. Applying more than the recommended rate can damage crops and increase safety risks. Always follow the label instructions.

Bad example:
Assistant: Pesticides work by disrupting plant cells and tissues...
"""

@st.cache_resource(show_spinner="Loading fine-tuned model...")
def load():
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                                device_map={"": 0}, torch_dtype=torch.bfloat16)
    ft = PeftModel.from_pretrained(base, ADAPTER)
    return tok, ft

def answer(history):
    msgs = [{"role": "system", "content": SYS}] + history[-6:]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    out = ft.generate(
    **enc,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.1,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,

    )
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

tok, ft = load()

# ============================================================
# UI
# ============================================================
st.markdown("""
<style>
.stApp { background:#121b15; color:#e6f2e8; }
.stApp p, .stApp li, .stApp span, .stApp label, .stApp h1, .stApp h2, .stApp h3 { color:#e6f2e8; }
[data-testid="stHeader"] { background:transparent; }
#MainMenu, [data-testid="stToolbar"], [data-testid="stStatusWidget"], [data-testid="stDecoration"], footer {
    visibility:hidden !important; height:0 !important; }
section[data-testid="stSidebar"] { background-color:#0c130e; border-right:1px solid #1f2e21; }
section[data-testid="stSidebar"] * { color:#dcebde !important; }
.hero { background: linear-gradient(135deg,#1f5c28,#3f8a4a); padding:26px; border-radius:20px;
        text-align:center; box-shadow:0 6px 18px rgba(0,0,0,0.45); margin-bottom:14px; }
.hero h1 { color:#fff !important; margin:0; font-size:2rem; }
.hero p  { color:#d9f5dd !important; margin:6px 0 0 0; }
.stChatMessage { background:#16201a; border:1px solid #233329; border-radius:16px;
                 box-shadow:0 2px 10px rgba(0,0,0,0.35); padding:10px 14px; }
.stChatMessage * { color:#e6f2e8 !important; }
.stButton>button { border-radius:12px; border:1px solid #3f8a4a; color:#7fd88f !important;
                    background:#16201a; font-weight:500; outline:none !important; box-shadow:none !important; }
.stButton>button:hover { background:#2e7d32; color:#fff !important; border-color:#2e7d32; }
.stButton>button:focus { outline:none !important; box-shadow:0 0 0 1px #3f8a4a !important; }
[data-testid="stMetric"] { background:#16201a; border:1px solid #233329; border-radius:14px;
                            padding:12px; box-shadow:0 2px 10px rgba(0,0,0,0.3); }
[data-testid="stMetric"] * { color:#e6f2e8 !important; }
[data-testid="stMetricLabel"] * { color:#9fc7a8 !important; }
[data-testid="stCaptionContainer"], .stCaption { color:#9fc7a8 !important; }
[data-testid="stAlertContentInfo"] { color:#d9f5dd !important; }
[data-testid="stAlert"] { background-color:#16241c !important; border:1px solid #2e7d32; }
[data-testid="stBottomBlockContainer"] { background-color:#121b15 !important; }
[data-testid="stBottom"] { background-color:#121b15 !important; }
[data-testid="stChatInput"] { background-color:#121b15 !important; border:1px solid #233329 !important;
                               border-radius:14px; outline:none !important; box-shadow:none !important; }
[data-testid="stChatInput"] > div { background-color:#121b15 !important; border:none !important;
                                     outline:none !important; box-shadow:none !important; }
[data-testid="stChatInput"] textarea { background-color:#121b15 !important; color:#e6f2e8 !important;
                                        border:none !important; outline:none !important; box-shadow:none !important; }
[data-testid="stChatInput"] textarea:focus { outline:none !important; box-shadow:none !important; }
[data-testid="stChatInput"]:focus-within { border-color:#3f8a4a !important; box-shadow:0 0 0 1px #3f8a4a !important; }
hr { border-color:#233329 !important; }
</style>
""", unsafe_allow_html=True)

USER_AVATAR, BOT_AVATAR = "🧑‍🌾", "🌱"

with st.sidebar:
    st.markdown("<h1 style='text-align:center;font-size:3rem;margin:0'>🌱</h1>", unsafe_allow_html=True)
    st.header("About")
    st.write("Ask anything about product usage, mixing ratios, application timing, or safety for our agricultural chemicals, microbial agents, and water-soluble fertilizers.")

    if st.button("Clear Chat", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

st.markdown("""
<div class="hero">
  <h1>🌱 BAMBOO TUNE </h1>
  <p>Agritech Customer Service Assistant</p>
</div>
""", unsafe_allow_html=True)

if "messages" not in st.session_state:
    st.session_state.messages = []
if "busy" not in st.session_state:
    st.session_state.busy = False

if not st.session_state.messages:
    st.info("👋 Hi! Ask me about product usage, mixing ratios, safety, or orders. I'll do my best to help.")

st.caption("💡 Quick questions")
b1, b2, b3 = st.columns(3)
preset = None
if b1.button(" How do I mix it?", use_container_width=True, disabled=st.session_state.busy): preset = "How do I mix the product with water?"
if b2.button(" How often to spray?", use_container_width=True, disabled=st.session_state.busy): preset = "How often should I apply the product?"
if b3.button(" Safe for vegetables?", use_container_width=True, disabled=st.session_state.busy): preset = "Is this product safe to use on vegetables?"

for m in st.session_state.messages:
    with st.chat_message(m["role"], avatar=USER_AVATAR if m["role"] == "user" else BOT_AVATAR):
        st.markdown(m["content"])

prompt = st.chat_input("Ask a product question...", disabled=st.session_state.busy) or preset

if prompt and not st.session_state.busy:
    st.session_state.busy = True
    try:
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user", avatar=USER_AVATAR):
            st.markdown(prompt)
        with st.chat_message("assistant", avatar=BOT_AVATAR):
            with st.spinner("Thinking..."):
                response = answer(st.session_state.messages)
            st.markdown(response)
        st.session_state.messages.append({"role": "assistant", "content": response})
    finally:
        st.session_state.busy = False


In [ ]:
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!streamlit run app.py &>/content/log.txt & ./cloudflared tunnel --url http://localhost:8501